Task 1: Read the CSV file in Pandas and create a DataFrame named Grc_df. What is the number of
rows and columns in Grc_df? Print the first 10 and last 10 rows of Grc_df. 


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

url = "Grocery_dataset.csv"

Grc_df = pd.read_csv(url)

rows, cols = Grc_df.shape

print(f"Rows {rows}, Columns: {cols} \n")
print(Grc_df.head(10))
print(Grc_df.tail(10))

Rows 5000, Columns: 12 

  Item_Identifier  Item_Weight Item_Fat_Content  Item_Visibility  \
0           FDA15        9.300          Low Fat         0.016047   
1           DRC01        5.920          Regular         0.019278   
2           FDN15       17.500          Low Fat         0.016760   
3           FDX07       19.200          Regular         0.000000   
4           NCD19        8.930          Low Fat         0.000000   
5           FDP36       10.395          Regular         0.000000   
6           FDO10       13.650          Regular         0.012741   
7           FDP10          NaN          Low Fat         0.127470   
8           FDH17       16.200          Regular         0.016687   
9           FDU28       19.200          Regular         0.094450   

               Item_Type  Item_MRP Outlet_Identifier  \
0                  Dairy  249.8092            OUT049   
1            Soft Drinks   48.2692            OUT018   
2                   Meat  141.6180            OUT049   
3 

Task 2: Are there any null values in the Grc_df? If yes, then in which columns and how many?
Finally, handle these null values using any strategy shown during the labs. 


In [2]:

Grc_df = Grc_df.replace(r'^\s*$', np.nan, regex=True)

print(Grc_df.isnull().sum())




Item_Identifier                 0
Item_Weight                   818
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  1439
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64


We have 818 null values in the Item weight column and 1439 null values in the Outlet size column. 

In [3]:

display(Grc_df['Item_Identifier'].unique())

Grc_df.groupby('Item_Identifier')['Item_Weight'].nunique()

    

<StringArray>
['FDA15', 'DRC01', 'FDN15', 'FDX07', 'NCD19', 'FDP36', 'FDO10', 'FDP10',
 'FDH17', 'FDU28',
 ...
 'FDP21', 'FDD33', 'DRC49', 'FDB10', 'FDS39', 'NCU29', 'NCP14', 'FDC48',
 'FDW14', 'FDS36']
Length: 1538, dtype: str

Item_Identifier
DRA12    1
DRA24    1
DRA59    1
DRB01    1
DRB13    1
        ..
NCZ30    1
NCZ41    1
NCZ42    1
NCZ53    1
NCZ54    1
Name: Item_Weight, Length: 1538, dtype: int64

We can see that all items under the same Item_Identifier have the same weight. With this we can assume that this is true for the items with null values 

In [4]:
Grc_df['Item_Weight'] = Grc_df.groupby('Item_Identifier')['Item_Weight'].transform(lambda x: x.fillna(x.mean()))

Grc_df.isnull().sum()

Item_Identifier                 0
Item_Weight                    35
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  1439
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

We have sucessfully filled most item weight by using the group average/mean values, however there asre still 35 vlaues that coul dnot be filled and we will drop these.

In [5]:
Grc_df = Grc_df.dropna(subset = ['Item_Weight'])

Grc_df.isnull().sum()

Item_Identifier                 0
Item_Weight                     0
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  1439
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

Now i have to clean the outlet_size values. I found that outlet_identifier could not be used, however i am going to try to use the outlet_type to normalise the nan values.

In [6]:
Grc_df.groupby('Outlet_Type')['Outlet_Size'].value_counts(dropna=False)

Grc_df['Outlet_Size'] = Grc_df.groupby('Outlet_Type')['Outlet_Size'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan))

Grc_df.isnull().sum()


Item_Identifier              0
Item_Weight                  0
Item_Fat_Content             0
Item_Visibility              0
Item_Type                    0
Item_MRP                     0
Outlet_Identifier            0
Outlet_Establishment_Year    0
Outlet_Size                  0
Outlet_Location_Type         0
Outlet_Type                  0
Item_Outlet_Sales            0
dtype: int64

Task 3: How many unique Outlet Sizes are there in the Grc_df? Which outlet size is maximum, and
which is minimum? 

In [7]:
Grc_df['Outlet_Size'].unique()

<StringArray>
['Medium', 'Small', 'High']
Length: 3, dtype: str

There are 3 unique sizes; Small, Medium and High. High is the maximum and small is the minimum.

Task 4: How many unique Item Fat Content types are in the Grc_df? List them. Do you see any
issues with the Item Fat Content types? If yes, then handle this issue. 


In [8]:
Grc_df['Item_Fat_Content'].unique()

<StringArray>
['Low Fat', 'Regular', 'low fat', 'LF', 'reg']
Length: 5, dtype: str

The column has inconsistent labels for the same categories. The values should be standarized to "Regular" and "Low Fat"

In [9]:
Grc_df['Item_Fat_Content'] = Grc_df['Item_Fat_Content'].replace({'LF': 'Low Fat', 'low fat': 'Low Fat', 'reg': 'Regular'})

Grc_df['Item_Fat_Content'].unique()

<StringArray>
['Low Fat', 'Regular']
Length: 2, dtype: str

Task 5: Drop the columns having index values of 0, 6 and create a new DataFrame Grc_new_df. 

In [10]:
Grc_new_df = Grc_df.drop(index=[0, 6])




Task 6: Using different Supermarket type listed in the column Outlet_Type create two different
DataFrames from Grc_new_df. Name these DataFrames as SupType_1 and SupType_2. 


In [11]:
SubType_1 = Grc_new_df[Grc_new_df['Outlet_Type'] == 'Supermarket Type1']

SubType_2 = Grc_new_df[Grc_new_df['Outlet_Type'] == 'Supermarket Type2']

display(SubType_1.head(10))

display(SubType_2.head(10))

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052
8,FDH17,16.20,Regular,0.016687,Frozen Foods,96.9726,OUT045,2002,Small,Tier 2,Supermarket Type1,1076.5986
9,FDU28,19.20,Regular,0.094450,Frozen Foods,187.8214,OUT017,2007,Small,Tier 2,Supermarket Type1,4710.5350
10,FDY07,11.80,Low Fat,0.000000,Fruits and Vegetables,45.5402,OUT049,1999,Medium,Tier 1,Supermarket Type1,1516.0266
11,FDA03,18.50,Regular,0.045464,Dairy,144.1102,OUT046,1997,Small,Tier 1,Supermarket Type1,2187.1530
12,FDX32,15.10,Regular,0.100014,Fruits and Vegetables,145.4786,OUT049,1999,Medium,Tier 1,Supermarket Type1,1589.2646
13,FDS46,17.60,Regular,0.047257,Snack Foods,119.6782,OUT046,1997,Small,Tier 1,Supermarket Type1,2145.2076
14,FDF32,16.35,Low Fat,0.068024,Fruits and Vegetables,196.4426,OUT013,1987,High,Tier 3,Supermarket Type1,1977.4260
15,FDP49,9.00,Regular,0.069089,Breakfast,56.3614,OUT046,1997,Small,Tier 1,Supermarket Type1,1547.3192


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
1,DRC01,5.920,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
5,FDP36,10.395,Regular,0.000000,Baking Goods,51.4008,OUT018,2009,Medium,Tier 3,Supermarket Type2,556.6088
16,NCB42,11.800,Low Fat,0.008596,Health and Hygiene,115.3492,OUT018,2009,Medium,Tier 3,Supermarket Type2,1621.8888
31,NCS17,18.600,Low Fat,0.080829,Health and Hygiene,96.4436,OUT018,2009,Medium,Tier 3,Supermarket Type2,2741.7644
32,FDP33,18.700,Low Fat,0.000000,Snack Foods,256.6672,OUT018,2009,Medium,Tier 3,Supermarket Type2,3068.0064
37,DRZ11,8.850,Regular,0.113124,Soft Drinks,122.5388,OUT018,2009,Medium,Tier 3,Supermarket Type2,1609.9044
43,FDC02,21.350,Low Fat,0.069103,Canned,259.9278,OUT018,2009,Medium,Tier 3,Supermarket Type2,6768.5228
55,FDK21,7.905,Low Fat,0.010053,Snack Foods,249.0408,OUT018,2009,Medium,Tier 3,Supermarket Type2,6258.5200
60,FDM20,10.000,Low Fat,0.000000,Fruits and Vegetables,246.9144,OUT018,2009,Medium,Tier 3,Supermarket Type2,3185.1872
82,FDV45,16.750,Low Fat,0.045231,Snack Foods,187.9556,OUT018,2009,Medium,Tier 3,Supermarket Type2,4693.8900


Task 7: Using Seaborn (“ggplot style”) create a (2,1) subplot of a box plot showing 5-point
summary of the column Item_MRP for SupType_1 and SupType_2. Which Outlet Type has
a higher median MRP? Are there any outliers? 